In [1]:
!mamba install pandas numpy scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 4.739 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ brotli-python       1.2.0       py313h8faef1e_3      emscripten-forge-4x
+ certifi             2026.10.23  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.28.0      pyh5ded981_0         conda-forge
+ pandas              3.1.0       np23py313h124dbc7_0  emscripten-forge-4x
+ pysocks             1.7.1       py313h1804a44_3      emscripten-forge-4x
+ requests            2.34.2      pyhcf101f3_0         conda-forge
+ scikit-

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [3]:
df = pd.read_csv("uber.csv")

print("First 5 Rows:")
display(df.head())

print("Dataset Shape:", df.shape)

First 5 Rows:


,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


Dataset Shape: (200000, 9)


In [4]:
print("Before Preprocessing:", df.shape)

print("Missing Values:")
print(df.isnull().sum())

df = df.dropna()

df = df.drop(
    ['Unnamed: 0', 'key', 'pickup_datetime'],
    axis=1,
    errors='ignore'
)

df = df[df['fare_amount'] > 0]
df = df[df['passenger_count'] > 0]

print("\nAfter Preprocessing:", df.shape)

display(df.head())

Before Preprocessing: (200000, 9)
Missing Values:
Unnamed: 0           0
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64

After Preprocessing: (199269, 6)


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,-73.925023,40.744085,-73.973082,40.761247,5


In [5]:
Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Lower Limit:", lower)
print("Upper Limit:", upper)

before = df.shape[0]

df = df[
    (df['fare_amount'] >= lower) &
    (df['fare_amount'] <= upper)
]

after = df.shape[0]

print("Outliers Removed:", before - after)
print("Dataset After Removing Outliers:", df.shape)

Lower Limit: -3.75
Upper Limit: 22.25
Outliers Removed: 17121
Dataset After Removing Outliers: (182148, 6)


In [6]:
correlation = df.corr(numeric_only=True)

print("Correlation Matrix:")
display(correlation)

Correlation Matrix:


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
fare_amount,1.000000,0.000125,0.002826,-0.002534,0.000715,0.011995
pickup_longitude,0.000125,1.000000,-0.811435,0.835878,-0.850520,-0.000582
pickup_latitude,0.002826,-0.811435,1.000000,-0.766797,0.683972,-0.001742
dropoff_longitude,-0.002534,0.835878,-0.766797,1.000000,-0.913666,0.000271
dropoff_latitude,0.000715,-0.850520,0.683972,-0.913666,1.000000,-0.000911
passenger_count,0.011995,-0.000582,-0.001742,0.000271,-0.000911,1.000000


In [7]:
X = df.drop('fare_amount', axis=1)
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

print("\nTraining Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

print("\nFirst 5 Training Records:")
display(X_train.head())

Training Data Shape: (145718, 5)
Testing Data Shape: (36430, 5)

Training Target Shape: (145718,)
Testing Target Shape: (36430,)

First 5 Training Records:


,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
165858,-73.971422,40.763980,-73.982404,40.739718,2
139654,-73.991973,40.725986,-74.005795,40.740529,1
171969,-73.991189,40.756032,-73.973810,40.764668,1
129197,-73.978095,40.752542,-73.965242,40.769197,1
199173,-73.980532,40.734073,-73.950061,40.776334,1


In [8]:
lr = LinearRegression()

lr.fit(X_train, y_train)

pred1 = lr.predict(X_test)

print("Linear Regression Model Trained Successfully")
print("First 5 Predictions:")
print(pred1[:5])

Linear Regression Model Trained Successfully
First 5 Predictions:
[8.92606993 8.9259355  8.92601777 8.9260894  8.9602252 ]


In [9]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred2 = rf.predict(X_test)

print("Random Forest Model Trained Successfully")
print("First 5 Predictions:")
print(pred2[:5])

Random Forest Model Trained Successfully
First 5 Predictions:
[ 8.376 12.276 11.806  5.621 10.915]


In [10]:
print("Linear Regression Results")
print("R2 Score:", r2_score(y_test, pred1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred1)))
print("MAE:", mean_absolute_error(y_test, pred1))

print("\nRandom Forest Regression Results")
print("R2 Score:", r2_score(y_test, pred2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred2)))
print("MAE:", mean_absolute_error(y_test, pred2))

Linear Regression Results
R2 Score: 0.00013528305667731555
RMSE: 4.129226610295564
MAE: 3.282610224143677

Random Forest Regression Results
R2 Score: 0.704646278522141
RMSE: 2.2442401071835367
MAE: 1.568954254598001


In [11]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R2 Score': [
        r2_score(y_test, pred1),
        r2_score(y_test, pred2)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, pred1)),
        np.sqrt(mean_squared_error(y_test, pred2))
    ],
    'MAE': [
        mean_absolute_error(y_test, pred1),
        mean_absolute_error(y_test, pred2)
    ]
})

display(results)

,Model,R2 Score,RMSE,MAE
0,Linear Regression,0.000135,4.129227,3.282610
1,Random Forest,0.704646,2.244240,1.568954


## Result

The Uber fare prediction model was implemented using Linear Regression and Random Forest Regression. The models were evaluated and compared using R² Score, RMSE, and MAE.

- Higher R² Score is better.
- Lower RMSE is better.
- Lower MAE is better.